In [102]:
import pandas as pd

# Data Acquisition

In [103]:
path = "../data/Resume.csv"

resume_df = pd.read_csv(path)

resume_df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [104]:
resume_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           2484 non-null   int64 
 1   Resume_str   2484 non-null   object
 2   Resume_html  2484 non-null   object
 3   Category     2484 non-null   object
dtypes: int64(1), object(3)
memory usage: 77.8+ KB


In [105]:
path = "../data/training_data.csv"

job_df = pd.read_csv(path)

job_df.head()

,company_name,job_description,position_title,description_length,model_response
0,Google,minimum qualifications\r\nbachelors degree or ...,Sales Specialist,2727,"{\r\n ""Core Responsibilities"": ""Responsible ..."
1,Apple,description\r\nas an asc you will be highly in...,Apple Solutions Consultant,828,"{\r\n ""Core Responsibilities"": ""as an asc yo..."
2,Netflix,its an amazing time to be joining netflix as w...,Licensing Coordinator - Consumer Products,3205,"{\r\n ""Core Responsibilities"": ""Help drive b..."
3,Robert Half,description\r\n\r\nweb designers looking to ex...,Web Designer,2489,"{\r\n ""Core Responsibilities"": ""Designing we..."
4,TrackFive,at trackfive weve got big goals were on a miss...,Web Developer,3167,"{\r\n ""Core Responsibilities"": ""Build and la..."


In [106]:
job_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 853 entries, 0 to 852
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   company_name        853 non-null    object
 1   job_description     853 non-null    object
 2   position_title      853 non-null    object
 3   description_length  853 non-null    int64 
 4   model_response      853 non-null    object
dtypes: int64(1), object(4)
memory usage: 33.4+ KB


In [107]:
job_df['model_response'][1]

' {\r\n  "Core Responsibilities": "as an asc you will be highly influential in growing mind and market share of apple products while building longterm relationships with those who share your passion customer experiences are driven through you and your partner team growing in an ever changing and challenging environment you strive for perfection whether its maintaining visual merchandising or helping to grow and develop your partner team",\r\n  "Required Skills": "a passion to help people understand how apple products can enrich their livesexcellent communication skills allowing you to be as comfortable in front of a small group as you are speaking with individuals years preferred working in a dynamic sales andor results driven environment as well as proven success developing customer loyaltyability to encourage a partner team and grow apple business",\r\n  "Educational Requirements": "N/A",\r\n  "Experience Level": "years preferred",\r\n  "Preferred Qualifications": "N/A",\r\n  "Compen

# Data Cleaning

### Missing Values

In [108]:
cols_with_question = resume_df.columns[resume_df.isin(['?']).any()].tolist()
print(cols_with_question)

[]


In [109]:
cols_with_question = job_df.columns[job_df.isin(['?']).any()].tolist()
print(cols_with_question)

[]


In [110]:
print( (job_df.isna().sum() / len(job_df) ) * 100)

company_name          0.0
job_description       0.0
position_title        0.0
description_length    0.0
model_response        0.0
dtype: float64


In [111]:
print( (resume_df.isna().sum() / len(resume_df) ) * 100)

ID             0.0
Resume_str     0.0
Resume_html    0.0
Category       0.0
dtype: float64


### Duplicates

In [112]:
job_df.duplicated().sum()

np.int64(0)

In [113]:
resume_df.duplicated().sum()

np.int64(0)

## Data Preprocessing

### Text Cleaning
- remove html
- remove whitespace
- remove extra chars
- lowercase

In [114]:
from bs4 import BeautifulSoup
import re

def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9+#.\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [115]:
job_df["clean_description"] = job_df["job_description"].apply(clean_text)
resume_df["clean_description"] = resume_df["Resume_str"].apply(clean_text)

In [116]:
print(job_df["clean_description"], '\n')
print(resume_df["clean_description"])

0      minimum qualifications bachelors degree or equ...
1      description as an asc you will be highly influ...
2      its an amazing time to be joining netflix as w...
3      description web designers looking to expand yo...
4      at trackfive weve got big goals were on a miss...
                             ...                        
848    job description parttime make big money at men...
849    responsibilities parkers internship program wa...
850    the borgen project is an innovative national c...
851    put the world on vacation at wyndham destinati...
852    this job handles customer inquiries by telepho...
Name: clean_description, Length: 853, dtype: object 

0       hr administrator marketing associate hr admini...
1       hr specialist us hr operations summary versati...
2       hr director summary over 20 years experience i...
3       hr specialist summary dedicated driven and dyn...
4       hr manager skill highlights hr skills hr depar...
                             

### Keyword Extraction
- skills
- responsibilites
- qualifications
- requirements
- degree
- experience

#### Resume

In [136]:
resume_df["clean_description"][7]

'hr manager professional summary senior hr professional with a continuous improvement approach to building and supporting organizations. driven hr manager bringing an innovative approach to human resource management while creating a team driven environment that fosters room for development and growth. dedicated hr professional with strong grasp of employment laws compliance issues and benefits plans. successfully introduces process improvements and staff development initiatives to drive corporate goal attainment. creative business partner dedicated to developing unique employee orientation and training programs that will generate a loyal and knowledgeable staff. skills administrative adp backup benefits bookkeeping clarify competitive customer service database delivery documentation employee relations filing government hr human resources human resource insurance job analysis labor relations law enforcement team building letters market meetings mail office payroll processing payroll per

In [118]:
def extract_degrees(text):
    if not isinstance(text, str):
        return []

    matches = re.findall(
        r"\b(?:high school diploma|associate(?:'s)?|bachelor(?:'s)?|master(?:'s)?|doctorate|ph\.?d\.?|mba)\b",
        text,
        flags=re.IGNORECASE
    )

    # dict for degrees
    degree_map = {
        "high school diploma": "High School Diploma",
        "associate": "Associate",
        "associate's": "Associate",
        "bachelor": "Bachelor",
        "bachelor's": "Bachelor",
        "master": "Master",
        "master's": "Master",
        "doctorate": "Doctorate",
        "phd": "Doctorate",
        "ph.d": "Doctorate",
        "ph.d.": "Doctorate",
        "mba": "MBA"
    }

    standardized = [
        degree_map.get(match.lower(), match.title())
        for match in matches
    ]

    return list(dict.fromkeys(standardized))

In [119]:
resume_df["degrees"] = resume_df["clean_description"].apply(extract_degrees)

In [120]:
resume_df["degrees"][0]

['Associate', 'High School Diploma']

#### Job

In [121]:
job_df["clean_description"][0]

'minimum qualifications bachelors degree or equivalent practical experience years of experience in saas or productivity tools businessexperience managing enterprise accounts with sales cycles preferred qualifications years of experience building strategic business partnerships with enterprise customersability to work through and with a reseller ecosystem to scale the businessability to plan pitch and execute a territory business strategyability to build relationships and to deliver results in a crossfunctionalmatrixed environmentability to identify crosspromoting and uppromoting opportunities within the existing account baseexcellent account management writtenverbal communication strategic and analyticalthinking skills about the job as a member of the google cloud team you inspire leading companies schools and government agencies to work smarter with google tools like google workspace search and chrome you advocate the innovative power of our products to make organizations more product

In [122]:
job_df['model_response'][0]

' {\r\n  "Core Responsibilities": "Responsible for expanding Google Workspace product adoption across an assigned territory. Build relationships with customers to understand needs and provide Google Workspace solutions. Partner with account teams to construct solutions and grow business for Google Workspace.",\r\n  "Required Skills": "Bachelor\'s degree or equivalent experience. Experience managing enterprise SaaS accounts and sales cycles.", \r\n  "Educational Requirements": "Bachelor\'s degree or equivalent experience.",\r\n  "Experience Level": "Experience managing enterprise SaaS accounts and sales cycles.",\r\n  "Preferred Qualifications": "Experience building strategic partnerships with enterprise customers. Ability to work through a reseller ecosystem. Excellent communication and strategic thinking skills.",\r\n  "Compensation and Benefits": "N/A"\r\n}'

In [123]:
import json

text = job_df['model_response'][0]
data = json.loads(text)

print(data["Core Responsibilities"])
print(data["Required Skills"])

Responsible for expanding Google Workspace product adoption across an assigned territory. Build relationships with customers to understand needs and provide Google Workspace solutions. Partner with account teams to construct solutions and grow business for Google Workspace.
Bachelor's degree or equivalent experience. Experience managing enterprise SaaS accounts and sales cycles.


In [124]:
parsed = job_df["model_response"].apply(json.loads)
parsed_df = pd.json_normalize(parsed)

job_df = job_df.join(parsed_df)

In [125]:
job_df["degrees"] = job_df["Educational Requirements"].apply(extract_degrees)

In [132]:
job_df["Educational Requirements"][:30]

0           Bachelor's degree or equivalent experience.
1                                                   N/A
2                                                   N/A
3                                                   N/A
4                                                   N/A
5                                                   N/A
6                                                   N/A
7                                                   N/A
8                                                   N/A
9           Bachelor's degree or equivalent experience.
10                                                  N/A
11          Bachelor's degree or equivalent experience.
12                                                  N/A
13    a bachelors degree in computer science, engine...
14                                                  N/A
15    Bachelor's degree in UI/UX or graphic design p...
16    Bachelor's degree required, MHA or MBA preferred.
17                                              

In [126]:
job_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 853 entries, 0 to 852
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   company_name               853 non-null    object
 1   job_description            853 non-null    object
 2   position_title             853 non-null    object
 3   description_length         853 non-null    int64 
 4   model_response             853 non-null    object
 5   clean_description          853 non-null    object
 6   Core Responsibilities      853 non-null    object
 7   Required Skills            853 non-null    object
 8   Educational Requirements   853 non-null    object
 9   Experience Level           853 non-null    object
 10  Preferred Qualifications   853 non-null    object
 11  Compensation and Benefits  853 non-null    object
 12  medical specialty          1 non-null      object
 13  schedule                   1 non-null      object
 14  license/ce

# Saving

In [127]:
resume_df.to_csv("../data/resume_clean.csv", index=False)
job_df.to_csv("../data/job_clean.csv", index=False)